# Recon DataPreProcessor Delay Troubleshooting Playbook

**Purpose:** Future troubleshooting reference for production delays between **DataMerger** and **DataPreProcessor** in the reconciliation pipeline.

This notebook is intentionally **not runnable end-to-end**. It is a VS Code–friendly notebook with copy-ready SQL and Bash snippets. Pick the relevant section, replace placeholders, and run commands in your company SQL client / edge node / Spark-YARN tools.

---

## Known context from current investigation

- Stack: Oracle metadata DB + CDH/Hadoop/HDFS + Apache Spark.
- Main evidence table: `FASTTRAC.OM_RECON_AUDIT_LOG`.
- Target example from investigation:
  - `RECON_ID = 8200065`
  - `COB_DATE = 20260427`
  - `DataMerger SUCCESS = 28-Apr-26 07:26:08 AM`
  - `DataPreProcessor SUCCESS = 28-Apr-26 01:58:56 PM`
  - Gap: about **6.55 hours / 392.8 minutes**.
- Correct runtime order:
  1. `DataMerger`
  2. `DataPreProcessor` / `CorePreProcessor`
  3. `ReconSchedule`
  4. `ReconLauncher`
  5. `ReconEngine - INITIATED`
  6. `ReconEngine - FINISHED`
- Important finding:
  - `DataPreProcessor` was running for other recons during the delay window.
  - One application-level DPP run had around **143 files and 140 recons**.
  - Therefore, the target recon may be healthy and only delayed because it was part of a larger shared DataPreProcessor batch.


## How to use this notebook

Replace these placeholders before running SQL/Bash:

```text
<<RECON_ID>>              Example: 8200065
<<COB_DATE_YYYYMMDD>>     Example: 20260427
<<SPARK_APPLICATION_ID>>  Example: application_1775994358481_893729
<<DPP_SOURCE_PATH>>       HDFS path from metadata
<<TARGETFILENAME>>        Target file from merger config
<<FEED_ID>>               Example: 1
```

Recommended flow:

1. Confirm audit timeline for one recon/COB.
2. Calculate the exact `DataMerger → DataPreProcessor` gap.
3. Extract the DataPreProcessor Spark application ID.
4. List all recons/files in the same application-level DPP batch.
5. Identify heavy contributors by file count, loaded count, record count, feed ID, and SLA status.
6. Check Feed 1 metadata and HDFS input volume.
7. Validate Spark/YARN start time, elapsed time, queue, and logs.
8. Decide whether this was:
   - Target-recon-specific
   - Feed-specific
   - Shared DPP batch surge
   - Spark/YARN queue/resource issue
   - Long transformation issue


## Current working hypothesis

Use this wording until evidence proves final RCA:

```text
DataPreProcessor was not globally down because it processed other recons during the same window.
The target recon appears to have been included in a larger shared DataPreProcessor Spark application that processed about 143 files and 140 recons.
Therefore, the target recon may not be the root cause. The delay may have been caused by a surge or long-running transformation from another recon/feed within the same DPP batch.
```

What to prove next:

```text
1. Did the DPP Spark application start early and run for hours?
2. Or was it submitted late?
3. Which recon/feed contributed the most file count, loaded count, record count, or SLA breach?
4. Was Feed 1 abnormal for this recon or another recon in the same app?
5. Was transformation logic the bottleneck?
```


## 1. Raw ordered audit timeline for one recon and COB date

This captures the true component order from the audit log.

```sql
WITH params AS (
    SELECT
        '<<RECON_ID>>'          AS p_recon_id,
        '<<COB_DATE_YYYYMMDD>>' AS p_cob_date
    FROM dual
),

events AS (
    SELECT
        comp_nm,
        recon_id,
        cob_date,
        start_time,
        status,
        record_count,
        message,

        CASE
            WHEN comp_nm = 'DataMerger' THEN 1
            WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor') THEN 2
            WHEN comp_nm = 'ReconSchedule' THEN 3
            WHEN comp_nm = 'ReconLauncher' THEN 4
            WHEN comp_nm = 'ReconEngine' AND status = 'INITIATED' THEN 5
            WHEN comp_nm = 'ReconEngine' AND status = 'FINISHED' THEN 6
            ELSE 99
        END AS stage_order
    FROM fasttrac.om_recon_audit_log a
    JOIN params p
        ON a.recon_id = p.p_recon_id
    WHERE REPLACE(TO_CHAR(a.cob_date), '-', '') = p.p_cob_date
      AND a.comp_nm IN (
            'DataMerger',
            'DataPreProcessor',
            'CorePreProcessor',
            'ReconSchedule',
            'ReconLauncher',
            'ReconEngine'
      )
)

SELECT
    stage_order,
    comp_nm,
    recon_id,
    cob_date,
    start_time,
    status,
    record_count,
    message
FROM events
ORDER BY start_time, stage_order;
```

Interpretation:

```text
Expected order:
DataMerger → DataPreProcessor → ReconSchedule → ReconLauncher → ReconEngine INITIATED → ReconEngine FINISHED
```


## 2. Gap calculation between stages

This calculates gaps between each stage. The key output is:

```text
DataMerger SUCCESS → DataPreProcessor SUCCESS
```

```sql
WITH params AS (
    SELECT
        '<<RECON_ID>>'          AS p_recon_id,
        '<<COB_DATE_YYYYMMDD>>' AS p_cob_date
    FROM dual
),

events AS (
    SELECT
        comp_nm,
        recon_id,
        REPLACE(TO_CHAR(cob_date), '-', '') AS cob_date_key,
        start_time,
        status,
        record_count,
        message,

        CASE
            WHEN comp_nm = 'DataMerger' THEN 1
            WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor') THEN 2
            WHEN comp_nm = 'ReconSchedule' THEN 3
            WHEN comp_nm = 'ReconLauncher' THEN 4
            WHEN comp_nm = 'ReconEngine' AND status = 'INITIATED' THEN 5
            WHEN comp_nm = 'ReconEngine' AND status = 'FINISHED' THEN 6
            ELSE 99
        END AS stage_order
    FROM fasttrac.om_recon_audit_log a
    JOIN params p
        ON a.recon_id = p.p_recon_id
    WHERE REPLACE(TO_CHAR(a.cob_date), '-', '') = p.p_cob_date
      AND a.comp_nm IN (
            'DataMerger',
            'DataPreProcessor',
            'CorePreProcessor',
            'ReconSchedule',
            'ReconLauncher',
            'ReconEngine'
      )
),

with_lag AS (
    SELECT
        recon_id,
        cob_date_key,
        comp_nm AS current_comp_nm,
        status AS current_status,
        start_time AS current_start_time,
        record_count,
        message,

        LAG(comp_nm) OVER (
            PARTITION BY recon_id, cob_date_key
            ORDER BY start_time, stage_order
        ) AS previous_comp_nm,

        LAG(status) OVER (
            PARTITION BY recon_id, cob_date_key
            ORDER BY start_time, stage_order
        ) AS previous_status,

        LAG(start_time) OVER (
            PARTITION BY recon_id, cob_date_key
            ORDER BY start_time, stage_order
        ) AS previous_start_time
    FROM events
)

SELECT
    recon_id,
    cob_date_key,

    previous_comp_nm,
    previous_status,
    previous_start_time,

    current_comp_nm,
    current_status,
    current_start_time,

    ROUND(
        (CAST(current_start_time AS DATE) - CAST(previous_start_time AS DATE)) * 24,
        2
    ) AS gap_hours,

    ROUND(
        (CAST(current_start_time AS DATE) - CAST(previous_start_time AS DATE)) * 24 * 60,
        2
    ) AS gap_minutes,

    record_count,
    message
FROM with_lag
WHERE previous_start_time IS NOT NULL
ORDER BY current_start_time;
```

Expected for the known example:

```text
previous_comp_nm = DataMerger
current_comp_nm  = DataPreProcessor
gap_hours        ≈ 6.55
gap_minutes      ≈ 392.8
```


## 3. Weekly summary for the same recon

Use this to see whether the same recon had repeated DPP delays this week.

```sql
WITH params AS (
    SELECT
        '<<RECON_ID>>' AS p_recon_id,
        TRUNC(SYSDATE, 'IW') AS week_start
    FROM dual
),

base AS (
    SELECT
        recon_id,
        REPLACE(TO_CHAR(cob_date), '-', '') AS cob_date_key,
        comp_nm,
        start_time,
        status,
        record_count,
        message
    FROM fasttrac.om_recon_audit_log a
    JOIN params p
        ON a.recon_id = p.p_recon_id
    WHERE a.start_time >= p.week_start
      AND a.comp_nm IN (
            'DataMerger',
            'DataPreProcessor',
            'CorePreProcessor',
            'ReconSchedule',
            'ReconLauncher',
            'ReconEngine'
      )
),

pivoted AS (
    SELECT
        recon_id,
        cob_date_key,

        MIN(CASE WHEN comp_nm = 'DataMerger'
                  AND status = 'SUCCESS'
                 THEN start_time END) AS datamerger_success_time,

        MIN(CASE WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
                  AND status = 'SUCCESS'
                 THEN start_time END) AS preprocessor_success_time,

        MIN(CASE WHEN comp_nm = 'ReconSchedule'
                  AND status = 'SUCCESS'
                 THEN start_time END) AS reconschedule_success_time,

        MIN(CASE WHEN comp_nm = 'ReconLauncher'
                  AND status = 'SUCCESS'
                 THEN start_time END) AS reconlauncher_success_time,

        MIN(CASE WHEN comp_nm = 'ReconEngine'
                  AND status = 'INITIATED'
                 THEN start_time END) AS reconengine_initiated_time,

        MAX(CASE WHEN comp_nm = 'ReconEngine'
                  AND status = 'FINISHED'
                 THEN start_time END) AS reconengine_finished_time,

        MAX(CASE WHEN comp_nm = 'DataMerger'
                 THEN record_count END) AS datamerger_record_count,

        MAX(CASE WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
                 THEN record_count END) AS preprocessor_record_count

    FROM base
    GROUP BY recon_id, cob_date_key
)

SELECT
    recon_id,
    cob_date_key,

    datamerger_success_time,
    preprocessor_success_time,
    reconschedule_success_time,
    reconlauncher_success_time,
    reconengine_initiated_time,
    reconengine_finished_time,

    datamerger_record_count,
    preprocessor_record_count,

    ROUND((CAST(preprocessor_success_time AS DATE) - CAST(datamerger_success_time AS DATE)) * 24, 2)
        AS gap_datamerger_to_preprocessor_hours,

    ROUND((CAST(preprocessor_success_time AS DATE) - CAST(datamerger_success_time AS DATE)) * 24 * 60, 2)
        AS gap_datamerger_to_preprocessor_minutes,

    ROUND((CAST(reconschedule_success_time AS DATE) - CAST(preprocessor_success_time AS DATE)) * 24 * 60, 2)
        AS gap_preprocessor_to_reconschedule_minutes,

    ROUND((CAST(reconlauncher_success_time AS DATE) - CAST(reconschedule_success_time AS DATE)) * 24 * 60, 2)
        AS gap_reconschedule_to_launcher_minutes,

    ROUND((CAST(reconengine_finished_time AS DATE) - CAST(reconengine_initiated_time AS DATE)) * 24 * 60, 2)
        AS reconengine_runtime_minutes

FROM pivoted
ORDER BY gap_datamerger_to_preprocessor_hours DESC NULLS LAST;
```

Interpretation:

```text
If only one COB date is delayed:
    likely input surge, feed-specific issue, or DPP app-level batching for that day.

If every COB date is delayed:
    likely recurring config, schedule, capacity, or DPP design issue.
```


## 4. Check whether DataPreProcessor was active for other recons during the target gap

This distinguishes global DPP outage from recon/application-specific delay.

```sql
WITH params AS (
    SELECT
        '<<RECON_ID>>'          AS p_recon_id,
        '<<COB_DATE_YYYYMMDD>>' AS p_cob_date
    FROM dual
),

target_window AS (
    SELECT
        MIN(CASE WHEN comp_nm = 'DataMerger'
                  AND status = 'SUCCESS'
                 THEN start_time END) AS datamerger_success_time,

        MIN(CASE WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
                  AND status = 'SUCCESS'
                 THEN start_time END) AS preprocessor_success_time
    FROM fasttrac.om_recon_audit_log a
    JOIN params p
        ON a.recon_id = p.p_recon_id
    WHERE REPLACE(TO_CHAR(a.cob_date), '-', '') = p.p_cob_date
)

SELECT
    a.comp_nm,
    a.recon_id,
    a.cob_date,
    a.start_time,
    a.status,
    a.record_count,
    a.message
FROM fasttrac.om_recon_audit_log a
CROSS JOIN target_window t
WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
  AND a.start_time >= t.datamerger_success_time
  AND a.start_time <= t.preprocessor_success_time
ORDER BY a.start_time;
```

Interpretation:

```text
If many other DPP rows appear:
    DataPreProcessor service was alive.
    Target recon delay is likely recon-specific or shared-batch-specific.

If no other DPP rows appear:
    Check DPP scheduler, Spark submit, YARN queue, platform outage.
```


## 5. Extract the DataPreProcessor Spark application ID

The application ID lets you analyze the full shared DPP batch.

```sql
SELECT
    comp_nm,
    recon_id,
    cob_date,
    start_time,
    status,
    record_count,
    REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
    message
FROM fasttrac.om_recon_audit_log
WHERE recon_id = '<<RECON_ID>>'
  AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
  AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
ORDER BY start_time;
```

Interpretation:

```text
If application ID is present:
    Use it for app-level RCA.

If application ID is not present:
    Search message text, job status table, scheduler logs, or YARN history by time window.
```


## 6. Summarize all recons/files in the same DPP Spark application

This validates whether the target recon was part of a large shared DPP batch.

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
)

SELECT
    t.spark_application_id,
    COUNT(*) AS audit_rows,
    COUNT(DISTINCT a.recon_id) AS recon_count,
    COUNT(DISTINCT REPLACE(TO_CHAR(a.cob_date), '-', '')) AS cob_date_count,
    MIN(a.start_time) AS first_dpp_audit_time,
    MAX(a.start_time) AS last_dpp_audit_time,
    SUM(NVL(a.record_count, 0)) AS total_record_count
FROM fasttrac.om_recon_audit_log a
JOIN target_app t
    ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
GROUP BY t.spark_application_id;
```

Interpretation:

```text
If recon_count ≈ 140 and files ≈ 143:
    The target recon was inside a large shared DPP application.
    The delay may be caused by batch-level surge, not by the target recon alone.
```


## 7. List every recon in the same DataPreProcessor app

Sort by `record_count` to locate heavy recons.

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
)

SELECT
    REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
    a.comp_nm,
    a.recon_id,
    REPLACE(TO_CHAR(a.cob_date), '-', '') AS cob_date_key,
    a.start_time,
    a.status,
    a.record_count,
    a.message
FROM fasttrac.om_recon_audit_log a
JOIN target_app t
    ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
ORDER BY
    a.record_count DESC NULLS LAST,
    a.start_time;
```

Interpretation:

```text
High record_count recons may be heavy transformation contributors.
A recon can delay the shared DPP app even if it has only one file but millions of records.
```


## 8. Join same-app recons to merged-status feed metadata

This identifies which recon/feed contributed high file count, loaded count, or SLA breach.

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
),

app_recons AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
        a.recon_id,
        REPLACE(TO_CHAR(a.cob_date), '-', '') AS cob_date_key,
        a.record_count,
        a.start_time AS dpp_success_time
    FROM fasttrac.om_recon_audit_log a
    JOIN target_app t
        ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
    WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
)

SELECT
    ar.spark_application_id,
    ar.recon_id,
    ar.cob_date_key,
    ar.dpp_success_time,
    ar.record_count AS dpp_record_count,

    ms.merge_feed_id,
    ms.file_count,
    ms.loaded_count,
    ms.issla_breached,
    ms.ismerged,
    ms.merged_ts,
    ms.modified_ts

FROM app_recons ar
LEFT JOIN fasttrac.om_recon_merged_status ms
    ON ms.reconid = ar.recon_id
   AND TO_CHAR(ms.triggered_cob_date, 'YYYYMMDD') = ar.cob_date_key

ORDER BY
    NVL(ms.file_count, 0) DESC,
    NVL(ms.loaded_count, 0) DESC,
    ar.record_count DESC NULLS LAST;
```

Interpretation:

```text
Top candidates for the surge:
- High FILE_COUNT
- High LOADED_COUNT
- High DPP_RECORD_COUNT
- ISSLA_BREACHED = Y
- MERGED_TS close to DPP completion
```


## 9. Find recons with more than one file in the same DPP app

Useful because the observed app had around 143 files and 140 recons.

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
),

app_recons AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
        a.recon_id,
        REPLACE(TO_CHAR(a.cob_date), '-', '') AS cob_date_key
    FROM fasttrac.om_recon_audit_log a
    JOIN target_app t
        ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
    WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
)

SELECT
    ar.spark_application_id,
    ar.recon_id,
    ar.cob_date_key,
    SUM(NVL(ms.file_count, 0)) AS total_file_count,
    SUM(NVL(ms.loaded_count, 0)) AS total_loaded_count,
    COUNT(DISTINCT ms.merge_feed_id) AS feed_count,
    MAX(ms.issla_breached) AS any_sla_breached,
    MAX(ms.merged_ts) AS latest_merged_ts
FROM app_recons ar
LEFT JOIN fasttrac.om_recon_merged_status ms
    ON ms.reconid = ar.recon_id
   AND TO_CHAR(ms.triggered_cob_date, 'YYYYMMDD') = ar.cob_date_key
GROUP BY
    ar.spark_application_id,
    ar.recon_id,
    ar.cob_date_key
HAVING SUM(NVL(ms.file_count, 0)) > 1
    OR SUM(NVL(ms.loaded_count, 0)) > 1
ORDER BY
    total_file_count DESC,
    total_loaded_count DESC;
```

Interpretation:

```text
Multiple files may contribute to listing/merge/preprocess overhead.
But also check record_count because one large file can be heavier than many small files.
```


## 10. Rank all recons in the DPP app by record count

This helps prove whether the target recon was normal or a heavy contributor.

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
),

app_recons AS (
    SELECT
        REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
        a.recon_id,
        REPLACE(TO_CHAR(a.cob_date), '-', '') AS cob_date_key,
        a.start_time,
        a.record_count,
        a.message
    FROM fasttrac.om_recon_audit_log a
    JOIN target_app t
        ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
    WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
),

ranked AS (
    SELECT
        ar.*,
        RANK() OVER (ORDER BY NVL(record_count, 0) DESC) AS record_count_rank,
        COUNT(*) OVER () AS total_recons_in_app,
        AVG(NVL(record_count, 0)) OVER () AS avg_record_count_in_app,
        MAX(NVL(record_count, 0)) OVER () AS max_record_count_in_app
    FROM app_recons ar
)

SELECT
    spark_application_id,
    recon_id,
    cob_date_key,
    start_time,
    record_count,
    record_count_rank,
    total_recons_in_app,
    ROUND(avg_record_count_in_app, 2) AS avg_record_count_in_app,
    max_record_count_in_app,
    message
FROM ranked
ORDER BY
    record_count_rank,
    start_time;
```

Interpretation:

```text
If target recon rank is low/average:
    Target recon was likely a victim of shared DPP batch latency.

If target recon rank is high:
    Target recon may be a contributor.
```


## 11. Identify Feed 1 metadata for the target recon

Use this to map `Feed 1` to actual file pattern / target file.

```sql
SELECT DISTINCT
    reconid,
    TO_CHAR(feed_id) AS feed_id,
    targetfilename
FROM reconmgmt.v_om_one_recon_merger_config
WHERE reconid = '<<RECON_ID>>'
ORDER BY
    TO_CHAR(feed_id),
    targetfilename;
```

Full Feed 1 metadata:

```sql
SELECT *
FROM reconmgmt.v_om_one_recon_merger_config
WHERE reconid = '<<RECON_ID>>'
  AND TO_CHAR(feed_id) = '1'
ORDER BY targetfilename;
```

Recon-level configuration:

```sql
SELECT *
FROM reconmgmt.v_om_one_recon_config
WHERE reconid = '<<RECON_ID>>';
```

Interpretation:

```text
Confirm whether "Feed 1" means FEED_ID = 1, left feed, right feed, or first configured feed.
Capture targetfilename, file pattern, source path, SLA, and date parsing fields.
```


## 12. Check Feed 1 merged status for the target recon

This confirms whether Feed 1 was abnormal for the target recon.

```sql
SELECT
    reconid,
    env,
    triggered_cob_date,
    merge_feed_id,
    file_count,
    loaded_count,
    issla_breached,
    ismerged,
    merged_ts,
    modified_ts
FROM fasttrac.om_recon_merged_status
WHERE reconid = '<<RECON_ID>>'
  AND TO_CHAR(triggered_cob_date, 'YYYYMMDD') = '<<COB_DATE_YYYYMMDD>>'
  AND TO_CHAR(merge_feed_id) = '1'
ORDER BY merged_ts;
```

Interpretation:

```text
If FILE_COUNT / LOADED_COUNT are normal and ISMERGED = Y early:
    Target recon Feed 1 was likely not the source of the delay.

If FILE_COUNT is high, LOADED_COUNT is high, SLA breached, or MERGED_TS is late:
    Feed 1 may be directly relevant to the delay.
```


## 13. Check Feed 1 for all recons in the same DPP app

This answers: "Was Feed 1 abnormal for another recon in the same batch?"

```sql
WITH target_app AS (
    SELECT DISTINCT
        REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id
    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND REPLACE(TO_CHAR(cob_date), '-', '') = '<<COB_DATE_YYYYMMDD>>'
      AND comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
      AND REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') IS NOT NULL
),

app_recons AS (
    SELECT DISTINCT
        a.recon_id,
        REPLACE(TO_CHAR(a.cob_date), '-', '') AS cob_date_key
    FROM fasttrac.om_recon_audit_log a
    JOIN target_app t
        ON REGEXP_SUBSTR(a.message, 'application_[0-9]+_[0-9]+') = t.spark_application_id
    WHERE a.comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
)

SELECT
    ms.reconid,
    TO_CHAR(ms.triggered_cob_date, 'YYYYMMDD') AS cob_date_key,
    ms.merge_feed_id,
    ms.file_count,
    ms.loaded_count,
    ms.issla_breached,
    ms.ismerged,
    ms.merged_ts,
    ms.modified_ts
FROM fasttrac.om_recon_merged_status ms
JOIN app_recons ar
    ON ar.recon_id = ms.reconid
   AND ar.cob_date_key = TO_CHAR(ms.triggered_cob_date, 'YYYYMMDD')
WHERE TO_CHAR(ms.merge_feed_id) = '1'
ORDER BY
    ms.file_count DESC,
    ms.loaded_count DESC,
    ms.merged_ts;
```

Interpretation:

```text
This helps identify if another recon's Feed 1 had the surge while the target recon was only impacted.
```


## 14. Compare Feed 1 volume across days for the target recon

Use this to show whether the delayed day was abnormal compared with normal days.

```sql
WITH audit_times AS (
    SELECT
        recon_id,
        REPLACE(TO_CHAR(cob_date), '-', '') AS cob_date_key,

        MIN(CASE WHEN comp_nm = 'DataMerger'
                  AND status = 'SUCCESS'
                 THEN start_time END) AS datamerger_success_time,

        MIN(CASE WHEN comp_nm IN ('DataPreProcessor', 'CorePreProcessor')
                  AND status = 'SUCCESS'
                 THEN start_time END) AS preprocessor_success_time

    FROM fasttrac.om_recon_audit_log
    WHERE recon_id = '<<RECON_ID>>'
      AND start_time >= TRUNC(SYSDATE, 'IW') - 7
    GROUP BY
        recon_id,
        REPLACE(TO_CHAR(cob_date), '-', '')
),

feed_status AS (
    SELECT
        reconid,
        TO_CHAR(triggered_cob_date, 'YYYYMMDD') AS cob_date_key,
        merge_feed_id,
        file_count,
        loaded_count,
        issla_breached,
        ismerged,
        merged_ts,
        modified_ts
    FROM fasttrac.om_recon_merged_status
    WHERE reconid = '<<RECON_ID>>'
      AND TO_CHAR(merge_feed_id) = '1'
      AND triggered_cob_date >= TRUNC(SYSDATE, 'IW') - 7
)

SELECT
    a.recon_id,
    a.cob_date_key,
    f.merge_feed_id,
    f.file_count,
    f.loaded_count,
    f.issla_breached,
    f.ismerged,
    f.merged_ts,
    f.modified_ts,

    a.datamerger_success_time,
    a.preprocessor_success_time,

    ROUND((CAST(a.preprocessor_success_time AS DATE) - CAST(a.datamerger_success_time AS DATE)) * 24, 2)
        AS gap_datamerger_to_preprocessor_hours,

    ROUND((CAST(a.preprocessor_success_time AS DATE) - CAST(a.datamerger_success_time AS DATE)) * 24 * 60, 2)
        AS gap_datamerger_to_preprocessor_minutes

FROM audit_times a
LEFT JOIN feed_status f
    ON f.reconid = a.recon_id
   AND f.cob_date_key = a.cob_date_key
ORDER BY
    gap_datamerger_to_preprocessor_hours DESC NULLS LAST;
```

Interpretation:

```text
If high Feed 1 volume correlates with high DPP delay:
    Supports surge-related RCA.

If Feed 1 is normal but delay is high:
    Look at other feeds/recons in the same DPP app.
```


## 15. Search DataPreProcessor transformation audit messages

Your manager mentioned long-running transformations in audit log. Use this to find transformation or error evidence.

```sql
SELECT
    comp_nm,
    recon_id,
    cob_date,
    start_time,
    status,
    record_count,
    REGEXP_SUBSTR(message, 'application_[0-9]+_[0-9]+') AS spark_application_id,
    message
FROM fasttrac.om_recon_audit_log
WHERE start_time >= TO_TIMESTAMP('2026-04-28 07:26:00', 'YYYY-MM-DD HH24:MI:SS')
  AND start_time <= TO_TIMESTAMP('2026-04-28 13:59:00', 'YYYY-MM-DD HH24:MI:SS')
  AND comp_nm IN (
        'DataPreProcessor',
        'CorePreProcessor',
        'DataConnector'
  )
  AND (
        recon_id = '<<RECON_ID>>'
     OR UPPER(message) LIKE '%<<RECON_ID>>%'
     OR UPPER(message) LIKE '%TRANSFORM%'
     OR UPPER(message) LIKE '%TRANSFORMATION%'
     OR UPPER(message) LIKE '%FILE LOADED%'
     OR UPPER(message) LIKE '%FILE%'
     OR UPPER(message) LIKE '%APPLICATION_%'
     OR UPPER(message) LIKE '%ERROR%'
     OR UPPER(message) LIKE '%FAILED%'
     OR UPPER(message) LIKE '%EXCEPTION%'
  )
ORDER BY start_time;
```

Adjust the timestamp window for your incident.

Interpretation:

```text
Shows whether DPP transformation:
- Started late
- Completed late
- Failed/retried
- Produced an application ID
- Logged transformation-specific messages
```


## 16. Find DPP source path and feed/file metadata

Use this to locate the HDFS path that DPP reads.

```sql
SELECT
    reconid,
    env,
    comp_id,
    param_name,
    param_value,
    created_ts,
    modified_ts,
    created_by,
    modified_by
FROM fasttrac.om_one_recon_meta_data_comp
WHERE reconid = '<<RECON_ID>>'
  AND (
        UPPER(comp_id) LIKE '%PREPROCESSOR%'
     OR UPPER(comp_id) LIKE '%DPP%'
     OR UPPER(param_name) LIKE '%DPP%'
     OR UPPER(param_name) LIKE '%HDFS%'
     OR UPPER(param_name) LIKE '%PATH%'
     OR UPPER(param_name) LIKE '%SOURCE%'
     OR UPPER(param_name) LIKE '%FILE%'
     OR UPPER(param_name) LIKE '%FEED%'
     OR UPPER(param_name) LIKE '%TRANSFORM%'
  )
ORDER BY
    comp_id,
    param_name;
```

Look for:

```text
dpp_src_file_hdfs_path
source path
feed path
file suffix
file pattern
transform config
```


## 17. HDFS checks for file surge

Run these on the edge node after you identify `<<DPP_SOURCE_PATH>>`.

Count files for the COB date:

```bash
hdfs dfs -ls -R <<DPP_SOURCE_PATH>> | grep <<COB_DATE_YYYYMMDD>> | wc -l
```

List files in the problem window:

```bash
hdfs dfs -ls -R <<DPP_SOURCE_PATH>> | awk '$6=="2026-04-28" && $7>="07:00" && $7<="14:00" {print}'
```

Count files in the problem window:

```bash
hdfs dfs -ls -R <<DPP_SOURCE_PATH>> | awk '$6=="2026-04-28" && $7>="07:00" && $7<="14:00" {print}' | wc -l
```

Get HDFS count:

```bash
hdfs dfs -count <<DPP_SOURCE_PATH>>
```

If file names contain recon/date/feed pattern:

```bash
hdfs dfs -ls -R <<DPP_SOURCE_PATH>> | grep <<RECON_ID>> | grep <<COB_DATE_YYYYMMDD>> | wc -l
```

For a known target file:

```bash
hdfs dfs -stat "%n|%y|%b" <<DPP_SOURCE_PATH>>/<<TARGETFILENAME>>
```

Interpretation:

```text
If HDFS file timestamp is early but DPP finishes late:
    DPP pickup/transformation/batch processing issue.

If file timestamp is late:
    File landing/visibility issue before DPP.

If file count surged compared with normal days:
    Supports surge-related RCA.
```


## 18. Spark/YARN application checks

Use the application ID from the audit log.

Application status:

```bash
yarn application -status <<SPARK_APPLICATION_ID>>
```

Key fields:

```text
Start-Time
Finish-Time
Elapsed-Time
State
Final-State
Queue
Tracking-URL
```

Application logs:

```bash
yarn logs -applicationId <<SPARK_APPLICATION_ID>> | grep -i -E "transform|transformation|file|recon|feed|<<RECON_ID>>|error|exception|failed|stage|task|spill|memory|timeout"
```

If logs are huge, write to file first:

```bash
yarn logs -applicationId <<SPARK_APPLICATION_ID>> > dpp_<<SPARK_APPLICATION_ID>>.log 2>&1
grep -i -E "transform|transformation|file|recon|feed|<<RECON_ID>>|error|exception|failed|stage|task|spill|memory|timeout" dpp_<<SPARK_APPLICATION_ID>>.log
```

Interpretation:

```text
Start-Time early, Finish-Time late:
    Long-running DPP app due to file/recon surge or transformation bottleneck.

Start-Time late, Finish-Time quick:
    DPP app was submitted late.

ACCEPTED for long time before RUNNING:
    YARN queue/resource delay.

RUNNING with long stages/tasks:
    transformation or skew issue.
```


## 19. Long-running application kill checklist

Only kill a long-running Spark/YARN application after approval and evidence capture.

Capture first:

```text
Application ID:
Recon IDs impacted:
Feed IDs impacted:
Start time:
Elapsed time:
YARN state:
Queue:
Current Spark stage/task symptom:
Manager approval:
```

Command:

```bash
yarn application -kill <<SPARK_APPLICATION_ID>>
```

Post-kill evidence:

```bash
yarn application -status <<SPARK_APPLICATION_ID>>
```

Notes:

```text
Do not kill only because duration is high.
Kill only when app is confirmed stuck/abnormal and business/manager approval is clear.
```


## 20. Check whether max-files-per-run or batch limit config exists

Your manager mentioned limiting the number of files picked in an individual run. First check if metadata already supports it.

Target recon:

```sql
SELECT
    reconid,
    env,
    comp_id,
    param_name,
    param_value,
    created_ts,
    modified_ts,
    created_by,
    modified_by
FROM fasttrac.om_one_recon_meta_data_comp
WHERE reconid = '<<RECON_ID>>'
  AND (
        UPPER(param_name) LIKE '%MAX%'
     OR UPPER(param_name) LIKE '%LIMIT%'
     OR UPPER(param_name) LIKE '%BATCH%'
     OR UPPER(param_name) LIKE '%COUNT%'
     OR UPPER(param_name) LIKE '%FILE%'
     OR UPPER(param_name) LIKE '%PICK%'
     OR UPPER(param_name) LIKE '%THRESHOLD%'
     OR UPPER(param_name) LIKE '%CHUNK%'
  )
ORDER BY
    comp_id,
    param_name;
```

Global search:

```sql
SELECT
    reconid,
    env,
    comp_id,
    param_name,
    param_value,
    created_ts,
    modified_ts
FROM fasttrac.om_one_recon_meta_data_comp
WHERE UPPER(param_name) LIKE '%MAX%'
   OR UPPER(param_name) LIKE '%LIMIT%'
   OR UPPER(param_name) LIKE '%BATCH%'
   OR UPPER(param_name) LIKE '%PICK%'
   OR UPPER(param_name) LIKE '%THRESHOLD%'
   OR UPPER(param_name) LIKE '%CHUNK%'
ORDER BY
    comp_id,
    param_name,
    reconid;
```

Interpretation:

```text
If config exists:
    Fix may be metadata/config-driven.

If no config exists:
    Fix may require code change in DPP file listing / batch slicing logic.
```


## 21. Schedule and gate flag check

Use this to inspect current gate state.

```sql
SELECT
    reconid,
    env,
    status,
    triggertime,
    daysofweek,
    frequency,
    last_trigger_cob_date,
    okfile,
    data_merge_ind,
    dpp_data_load_ind,
    modified_ts,
    modified_by,
    created_ts
FROM fasttrac.om_recon_schedule
WHERE reconid = '<<RECON_ID>>'
ORDER BY modified_ts DESC;
```

Important caution:

```text
This table may show only current state because indicators can be reset after OK file creation.
For historical proof, use audit log or Oracle flashback if enabled.
```


## 22. Optional: Oracle Flashback for historical schedule flags

Use only if Flashback Query is enabled.

Example: state shortly after DataMerger success.

```sql
SELECT
    reconid,
    env,
    status,
    triggertime,
    daysofweek,
    frequency,
    last_trigger_cob_date,
    okfile,
    data_merge_ind,
    dpp_data_load_ind,
    modified_ts,
    modified_by
FROM fasttrac.om_recon_schedule
AS OF TIMESTAMP TO_TIMESTAMP('2026-04-28 07:31:00', 'YYYY-MM-DD HH24:MI:SS')
WHERE reconid = '<<RECON_ID>>';
```

Example: state just before DataPreProcessor success.

```sql
SELECT
    reconid,
    env,
    status,
    triggertime,
    daysofweek,
    frequency,
    last_trigger_cob_date,
    okfile,
    data_merge_ind,
    dpp_data_load_ind,
    modified_ts,
    modified_by
FROM fasttrac.om_recon_schedule
AS OF TIMESTAMP TO_TIMESTAMP('2026-04-28 13:55:00', 'YYYY-MM-DD HH24:MI:SS')
WHERE reconid = '<<RECON_ID>>';
```

Interpretation:

```text
DATA_MERGE_IND = Y early but DPP late:
    DPP pickup/batch/transformation issue.

DATA_MERGE_IND becomes Y late:
    Merger-to-schedule handoff issue.

DPP_DATA_LOAD_IND becomes Y only at the end:
    Normal if it marks completion rather than start.
```


## 23. Optional: Oracle object discovery if code search does not find table names

Find synonyms/views:

```sql
SELECT
    owner,
    synonym_name,
    table_owner,
    table_name
FROM all_synonyms
WHERE table_name IN (
    'OM_RECON_DATAMERGE',
    'OM_RECON_DATAMERGE_DETAIL',
    'OM_RECON_MERGED_STATUS',
    'OM_RECON_RUN_STATUS',
    'OM_RECON_SCHEDULE',
    'OM_RECON_SCHEDULE_CAL',
    'RECONCILIATIONRUN',
    'OM_RECON_AUDIT_LOG'
)
ORDER BY owner, synonym_name;
```

Find dependent packages/views/procedures:

```sql
SELECT
    owner,
    name,
    type,
    referenced_owner,
    referenced_name,
    referenced_type
FROM all_dependencies
WHERE referenced_name IN (
    'OM_RECON_DATAMERGE',
    'OM_RECON_DATAMERGE_DETAIL',
    'OM_RECON_MERGED_STATUS',
    'OM_RECON_RUN_STATUS',
    'OM_RECON_SCHEDULE',
    'OM_RECON_SCHEDULE_CAL',
    'RECONCILIATIONRUN',
    'OM_RECON_AUDIT_LOG'
)
ORDER BY referenced_name, owner, type, name;
```

Search source:

```sql
SELECT
    owner,
    name,
    type,
    line,
    text
FROM all_source
WHERE UPPER(text) LIKE '%OM_RECON_DATAMERGE%'
   OR UPPER(text) LIKE '%OM_RECON_MERGED_STATUS%'
   OR UPPER(text) LIKE '%OM_RECON_RUN_STATUS%'
   OR UPPER(text) LIKE '%OM_RECON_SCHEDULE%'
   OR UPPER(text) LIKE '%OM_RECON_AUDIT_LOG%'
   OR UPPER(text) LIKE '%RECONCILIATIONRUN%'
ORDER BY owner, name, type, line;
```


## 24. Cursor / repo search prompt

Use this in Cursor if table names do not appear directly in code.

```text
The exact Oracle physical table names may not appear directly in the repo.
Search using config keys, dynamic table names, column names, and process terms.

Context:
Recon pipeline with Oracle metadata DB, CDH/Hadoop/HDFS, Apache Spark.
Delay between DataMerger and DataPreProcessor.
Audit log shows DataPreProcessor app may process many recons/files in one shared batch.

Search terms:
- datamerge
- data_merge
- preprocessor
- preprocess
- CorePreProcessor
- DataPreProcessor
- merged_status
- recon_run_status
- audit_log
- reconid
- merged_ts
- ismerged
- merge_feed_id
- triggered_cob_date
- file_count
- loaded_count
- data_merge_ind
- dpp_data_load_ind
- jobid
- application_
- SparkFinalState
- yarn
- transform
- transformation
- max files
- file limit
- batch limit
- chunk size
- file pickup

Tasks:
1. Identify DPP file listing and pickup logic.
2. Identify whether DPP processes multiple recons in one Spark app.
3. Identify where max file count per run can be configured or added.
4. Identify transformation logic and audit log writes.
5. Identify SQL or DataFrame filters for recon eligibility.
6. Identify where application ID is written to audit log.
7. Return file path, method/function, matching lines, and config keys.
```


## 25. Manager update template

Use this after collecting evidence.

```text
I validated that DataPreProcessor was active during the delay window, so this does not appear to be a global DPP outage.

For RECON_ID <<RECON_ID>> / COB <<COB_DATE_YYYYMMDD>>, DataMerger completed at <time>, while DataPreProcessor completed at <time>, producing a gap of <minutes> minutes.

Additional log analysis shows the DataPreProcessor Spark application was shared across approximately <N> files and <M> recons. That means <<RECON_ID>> may not be the root cause; it may have been delayed as part of a larger DPP batch.

I am checking the same application-level batch to identify the heavy recon/feed contributors using:
1. Record count per recon
2. Feed file_count / loaded_count
3. SLA breach flag
4. Feed 1 metadata
5. DPP transformation messages
6. YARN/Spark elapsed time and logs

Current working hypothesis:
The delay is likely caused by shared DPP batch surge or long-running transformation from one or more feeds/recons, rather than an issue isolated to <<RECON_ID>>.
```


## 26. RCA decision tree

```text
Case 1:
DPP Spark app Start-Time early, Finish-Time late
AND app contains many recons/files
→ Shared DPP batch surge or long transformation.

Case 2:
DPP Spark app Start-Time late, Finish-Time quick
→ Scheduler/pickup delay.

Case 3:
DPP Spark app stuck in ACCEPTED before RUNNING
→ YARN queue/resource delay.

Case 4:
Target recon Feed 1 has normal file_count/loaded_count
BUT another recon/feed in same app has high volume
→ Target recon is impacted, not root cause.

Case 5:
Target recon Feed 1 has high file_count/loaded_count or SLA breach
→ Target Feed 1 may be contributor.

Case 6:
Merged status is late
→ Issue before DPP pickup, possibly merger/feed readiness.

Case 7:
Merged status early but DPP completion late
→ DPP pickup/app-level processing/transformation issue.
```
